In [156]:
import logging
from pathlib import Path
import re
import subprocess

import pandas as pd

In [152]:
logger = logging.getLogger()

In [49]:
! ls /c/Users/sesti/Music/Playlists/djutil/test/src

1.DÃŠTRE - Ten Bands.mp3
1.huck.jorris, Saturna - promise (saturna remix).mp3
2.Sub Basics - Shipment Pt.2.mp3
Bass B$tch.mp3
Criso & Breathworkz - Wicked_2051092604 - Criso.mp3
Dalek One - 913 (Subs Go Brrr)_1257792283 - 19K.m4a


In [50]:
src_dir = Path("C:/Users/sesti/Music/Playlists/djutil/test/src")
dst_dir = Path("C:/Users/sesti/Music/Playlists/djutil/test/dst")

In [148]:
def get_file_df(dir_path, glob_pat="*"):
    path = []
    parent = []
    name = []
    stem = []
    suffix = []
    for pp in dir_path.glob(glob_pat):
        path.append(pp.resolve())
        parent.append(pp.parent)
        name.append(pp.name)
        stem.append(pp.stem)
        suffix.append(pp.suffix)
    df = pd.DataFrame({"path": path, "parent": parent, "name": name, "stem": stem, "suffix": suffix})
    return df

In [82]:
def strip_digit_prefix(stem):
    # "2.Sub Basics - Shipment Pt.2" => "Sub Basics - Shipment Pt.2"
    return re.sub(r'^\d+.', '', stem)

In [85]:
df.stem.map(strip_digit_prefix)

0                                  DÊTRE - Ten Bands
1     huck.jorris, Saturna - promise (saturna remix)
2                         Sub Basics - Shipment Pt.2
3                                         Bass B$tch
4    Criso & Breathworkz - Wicked_2051092604 - Criso
5    Dalek One - 913 (Subs Go Brrr)_1257792283 - 19K
Name: stem, dtype: str

In [87]:
def strip_underscore_digits(stem):
    # "Criso & Breathworkz - Wicked_2051092604 - Criso" => "Criso & Breathworkz - Wicked - Criso"
    return re.sub(r'_\d+', '', stem)

In [89]:
df.stem.map(strip_underscore_digits)

0                                 1.DÊTRE - Ten Bands
1    1.huck.jorris, Saturna - promise (saturna remix)
2                        2.Sub Basics - Shipment Pt.2
3                                          Bass B$tch
4                Criso & Breathworkz - Wicked - Criso
5                Dalek One - 913 (Subs Go Brrr) - 19K
Name: stem, dtype: str

In [91]:
df.stem.map(strip_digit_prefix).map(strip_underscore_digits)

0                                 DÊTRE - Ten Bands
1    huck.jorris, Saturna - promise (saturna remix)
2                        Sub Basics - Shipment Pt.2
3                                        Bass B$tch
4              Criso & Breathworkz - Wicked - Criso
5              Dalek One - 913 (Subs Go Brrr) - 19K
Name: stem, dtype: str

In [164]:
df = get_file_df(src_dir)
df["new_stem"] = df.stem.map(strip_digit_prefix).map(strip_underscore_digits)
df["new_suffix"] = ".wav"
df["new_parent"] = dst_dir
df["new_sample_rate"] = str(44100)
df["new_sample_fmt"] = "s16"
df["new_path"] = df.apply(lambda row: row.new_parent / (row.new_stem + row.new_suffix), axis=1).values


In [165]:
df

,path,parent,name,stem,suffix,new_stem,new_suffix,new_parent,new_sample_rate,new_sample_fmt,new_path
0,C:\Users\sesti\Music\Playlists\djutil\test\src...,C:\Users\sesti\Music\Playlists\djutil\test\src,1.DÊTRE - Ten Bands.mp3,1.DÊTRE - Ten Bands,.mp3,DÊTRE - Ten Bands,.wav,C:\Users\sesti\Music\Playlists\djutil\test\dst,44100,s16,C:\Users\sesti\Music\Playlists\djutil\test\dst...
1,C:\Users\sesti\Music\Playlists\djutil\test\src...,C:\Users\sesti\Music\Playlists\djutil\test\src,"1.huck.jorris, Saturna - promise (saturna remi...","1.huck.jorris, Saturna - promise (saturna remix)",.mp3,"huck.jorris, Saturna - promise (saturna remix)",.wav,C:\Users\sesti\Music\Playlists\djutil\test\dst,44100,s16,C:\Users\sesti\Music\Playlists\djutil\test\dst...
2,C:\Users\sesti\Music\Playlists\djutil\test\src...,C:\Users\sesti\Music\Playlists\djutil\test\src,2.Sub Basics - Shipment Pt.2.mp3,2.Sub Basics - Shipment Pt.2,.mp3,Sub Basics - Shipment Pt.2,.wav,C:\Users\sesti\Music\Playlists\djutil\test\dst,44100,s16,C:\Users\sesti\Music\Playlists\djutil\test\dst...
3,C:\Users\sesti\Music\Playlists\djutil\test\src...,C:\Users\sesti\Music\Playlists\djutil\test\src,Bass B$tch.mp3,Bass B$tch,.mp3,Bass B$tch,.wav,C:\Users\sesti\Music\Playlists\djutil\test\dst,44100,s16,C:\Users\sesti\Music\Playlists\djutil\test\dst...
4,C:\Users\sesti\Music\Playlists\djutil\test\src...,C:\Users\sesti\Music\Playlists\djutil\test\src,Criso & Breathworkz - Wicked_2051092604 - Cris...,Criso & Breathworkz - Wicked_2051092604 - Criso,.mp3,Criso & Breathworkz - Wicked - Criso,.wav,C:\Users\sesti\Music\Playlists\djutil\test\dst,44100,s16,C:\Users\sesti\Music\Playlists\djutil\test\dst...
5,C:\Users\sesti\Music\Playlists\djutil\test\src...,C:\Users\sesti\Music\Playlists\djutil\test\src,Dalek One - 913 (Subs Go Brrr)_1257792283 - 19...,Dalek One - 913 (Subs Go Brrr)_1257792283 - 19K,.m4a,Dalek One - 913 (Subs Go Brrr) - 19K,.wav,C:\Users\sesti\Music\Playlists\djutil\test\dst,44100,s16,C:\Users\sesti\Music\Playlists\djutil\test\dst...


In [180]:
def run_ffmpeg(row):
    assert row.path
    assert row.new_sample_fmt
    assert row.new_sample_rate
    assert row.new_path
    if row.new_path.is_file():
        logger.warning(f"File exists, skipping: {row.new_path}")
        return None
    else:
        command = [
            'ffmpeg',
            '-i', row.path,
            '-sample_fmt', row.new_sample_fmt,
            '-ar', row.new_sample_rate,
            row.new_path
        ]
        process = subprocess.Popen(command)
        return_code = process.wait()
        return int(return_code)
        
    

In [186]:
df.apply(run_ffmpeg, axis=1)

0    0
1    0
2    0
3    0
4    0
5    0
dtype: int64

In [185]:
! rm /c/Users/sesti/Music/Playlists/djutil/test/dst/*